# Day 050 Project: Build the Insight Engine

## What You're Building

The Section 3 capstone: run the complete InsightEngine on the retail sales dataset and produce a full AI-narrated report.

## Project Requirements

1. Create `engine = InsightEngine(title='Retail Sales 2023')`
2. Call `result = engine.run(df, target_col='revenue')`    where `df = make_sample_data(300)`
3. Save a 2×2 dashboard to `insight_report.png`
4. Print `engine.narrative` (the AI executive summary)
5. Print the model report (CV R², test R², RMSE, top feature)
6. Run `_run_project_checks()` to verify

## Bonus Challenges

- Try the engine on a real CSV you have (or download one from Kaggle)
- Extend `save_chart` with a 5th panel: time series of revenue by week   (requires the date column to be set as the index with `parse_time_series`)
- Add a `compare_models` method that trains both LinearRegression and   a DecisionTreeRegressor and returns the better one by CV R²

## Provided: All Implementations

In [ ]:
import io
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import ollama
warnings.filterwarnings('ignore')


def make_sample_data(n: int = 300, seed: int = 42) -> pd.DataFrame:
    """
    Retail sales dataset.
    Columns: date (str), region, category, units_sold, price, discount, revenue.
    Revenue = units_sold*4 + price*1.5 - discount*150 + noise (5% nulls injected).
    """
    rng      = np.random.default_rng(seed)
    dates    = pd.date_range('2023-01-01', periods=n, freq='D').strftime('%Y-%m-%d')
    region   = rng.choice(['North', 'South', 'East', 'West'], n)
    category = rng.choice(['Electronics', 'Clothing', 'Food', 'Books'], n)
    units    = rng.integers(1, 50, n)
    price    = rng.uniform(5.0, 200.0, n).round(2)
    discount = rng.choice([0.0, 0.05, 0.10, 0.15, 0.20], n)
    revenue  = (units * 4.0 + price * 1.5 - discount * 150
                + rng.standard_normal(n) * 20).round(2)
    null_idx = rng.choice(n, size=max(1, int(n * 0.05)), replace=False)
    revenue  = revenue.astype(float)
    revenue[null_idx] = np.nan
    return pd.DataFrame({
        'date':       pd.Series(dates),
        'region':     region,
        'category':   category,
        'units_sold': units,
        'price':      price,
        'discount':   discount,
        'revenue':    revenue,
    })


def load_and_clean(source) -> pd.DataFrame:
    """
    Load from CSV string / file path / DataFrame and clean.

    Steps applied in order:
      1. Parse source into a DataFrame
      2. Detect and parse date/time columns to datetime64
      3. Fill numeric NaN with column median
      4. Drop exact duplicate rows
    """
    if isinstance(source, pd.DataFrame):
        df = source.copy()
    elif isinstance(source, str) and ('\n' in source or ',' in source[:200]):
        df = pd.read_csv(io.StringIO(source))
    else:
        df = pd.read_csv(source)

    # Detect date columns by name
    for col in df.columns:
        if any(kw in col.lower() for kw in ('date', 'time', 'created', 'updated')):
            try:
                df[col] = pd.to_datetime(df[col], errors='coerce')
            except Exception:
                pass

    # Fill numeric NaN with column median
    for col in df.select_dtypes(include='number').columns:
        median = df[col].median()
        df[col] = df[col].fillna(median)

    # Drop duplicates
    df = df.drop_duplicates().reset_index(drop=True)
    return df


def run_eda(df: pd.DataFrame) -> dict:
    """
    Compute an EDA summary dict with keys:
        shape, columns, dtypes, null_counts,
        numeric_summary, correlations, category_counts,
        numeric_cols, cat_cols
    """
    num_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.tolist()

    numeric_summary = {}
    for col in num_cols:
        s = df[col].dropna()
        numeric_summary[col] = {
            'mean':   round(float(s.mean()),   4),
            'std':    round(float(s.std()),    4),
            'min':    round(float(s.min()),    4),
            'max':    round(float(s.max()),    4),
            'median': round(float(s.median()), 4),
        }

    correlations = {}
    if len(num_cols) >= 2:
        cm = df[num_cols].corr()
        for col in num_cols:
            correlations[col] = {
                other: round(float(cm.loc[col, other]), 4)
                for other in num_cols if other != col
            }

    category_counts = {
        col: df[col].value_counts().head(10).to_dict()
        for col in cat_cols
    }

    return {
        'shape':           {'rows': int(df.shape[0]), 'cols': int(df.shape[1])},
        'columns':         df.columns.tolist(),
        'dtypes':          {c: str(t) for c, t in df.dtypes.items()},
        'null_counts':     df.isnull().sum().to_dict(),
        'numeric_summary': numeric_summary,
        'correlations':    correlations,
        'category_counts': category_counts,
        'numeric_cols':    num_cols,
        'cat_cols':        cat_cols,
    }


def train_and_evaluate(df: pd.DataFrame, target_col: str,
                        test_size: float = 0.2,
                        random_state: int = 42) -> dict:
    """
    Auto-select numeric features, scale, train LinearRegression with 5-fold CV,
    and evaluate on a held-out test set.

    Returns dict with keys:
        target, features, cv_r2 (mean/std), test_r2, test_rmse, test_mae,
        coefficients (feature → value), n_train, n_test
    """
    if target_col not in df.columns:
        return {'error': f'target column {target_col!r} not found'}

    num_cols     = df.select_dtypes(include='number').columns.tolist()
    feature_cols = [c for c in num_cols if c != target_col]

    if not feature_cols:
        return {'error': 'no numeric feature columns found'}

    sub   = df[feature_cols + [target_col]].dropna()
    X     = sub[feature_cols]
    y     = sub[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    scaler   = StandardScaler()
    X_tr_s   = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
    X_te_s   = pd.DataFrame(scaler.transform(X_test),      columns=feature_cols)

    model = LinearRegression()
    kf    = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_tr_s, y_train, cv=kf, scoring='r2')

    model.fit(X_tr_s, y_train)
    y_pred = model.predict(X_te_s)

    return {
        'target':    target_col,
        'features':  feature_cols,
        'cv_r2':     {'mean': round(float(cv_scores.mean()), 4),
                      'std':  round(float(cv_scores.std()),  4)},
        'test_r2':   round(float(r2_score(y_test, y_pred)),                         4),
        'test_rmse': round(float(np.sqrt(mean_squared_error(y_test, y_pred))),       2),
        'test_mae':  round(float(mean_absolute_error(y_test, y_pred)),               2),
        'coefficients': {
            col: round(float(c), 4)
            for col, c in zip(feature_cols, model.coef_)
        },
        'n_train': int(len(X_train)),
        'n_test':  int(len(X_test)),
    }


def narrate_insights(eda: dict, model_report: dict,
                     title: str = 'Dataset',
                     model: str = 'llama3.2') -> str:
    """
    Generate a 3-sentence executive summary via Ollama (llama3.2).
    Falls back to a structured plain-text summary if Ollama is unavailable.
    """
    shape    = eda.get('shape', {})
    num_cols = eda.get('numeric_cols', [])
    cat_cols = eda.get('cat_cols', [])

    context_parts = [
        f"Dataset: {title}",
        f"Shape: {shape.get('rows', '?')} rows \u00d7 {shape.get('cols', '?')} columns",
        f"Numeric columns: {', '.join(num_cols) if num_cols else 'none'}",
        f"Categorical columns: {', '.join(cat_cols) if cat_cols else 'none'}",
    ]

    num_summary = eda.get('numeric_summary', {})
    for col, stats in list(num_summary.items())[:4]:
        context_parts.append(
            f"  {col}: mean={stats['mean']:.2f}, std={stats['std']:.2f}, "
            f"min={stats['min']:.2f}, max={stats['max']:.2f}"
        )

    if model_report and 'test_r2' in model_report:
        context_parts.append(
            f"LinearRegression predicting {model_report['target']}: "
            f"CV R\u00b2={model_report['cv_r2']['mean']:.4f} \u00b1 {model_report['cv_r2']['std']:.4f}, "
            f"Test R\u00b2={model_report['test_r2']:.4f}, "
            f"RMSE={model_report['test_rmse']:.2f}"
        )
        top = sorted(model_report.get('coefficients', {}).items(),
                     key=lambda kv: abs(kv[1]), reverse=True)
        if top:
            context_parts.append(
                f"Strongest predictor: {top[0][0]} (coef={top[0][1]:.4f})"
            )

    context = '\n'.join(context_parts)
    prompt  = (
        f"You are a concise data analyst. Write a 3-sentence executive summary "
        f"for a non-technical stakeholder based on this analysis:\n\n{context}\n\n"
        f"Cover: (1) what the data contains, (2) the key pattern or insight, "
        f"(3) one concrete recommendation."
    )

    try:
        response = ollama.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
        )
        return response['message']['content'].strip()
    except Exception:
        lines = [f"Analysis of {title}: {shape.get('rows', '?')} rows, "
                 f"{shape.get('cols', '?')} columns."]
        if num_cols:
            lines.append(f"Numeric features: {', '.join(num_cols)}.")
        if model_report and 'test_r2' in model_report:
            lines.append(
                f"Predictive model (LinearRegression \u2192 {model_report['target']}): "
                f"Test R\u00b2={model_report['test_r2']:.4f}, "
                f"RMSE={model_report['test_rmse']:.2f}."
            )
        return ' '.join(lines)


class InsightEngine:
    """
    End-to-end data insight pipeline for Section 3 capstone.

    Usage:
        engine = InsightEngine(title='Retail Sales 2023')
        result = engine.run(df_or_csv, target_col='revenue')
        engine.save_chart('dashboard.png')
        print(engine.narrative)
    """

    def __init__(self, title: str = 'Dataset', llm_model: str = 'llama3.2'):
        self.title        = title
        self.llm_model    = llm_model
        self.df           = None
        self.eda          = None
        self.model_report = None
        self.narrative    = None

    def run(self, source, target_col: str = None) -> dict:
        """
        Full pipeline:
          1. load_and_clean(source)
          2. run_eda(df)
          3. train_and_evaluate(df, target_col)  — if target_col provided
          4. narrate_insights(eda, model_report)

        Returns dict with keys: df, eda, model_report, narrative
        """
        self.df           = load_and_clean(source)
        self.eda          = run_eda(self.df)
        self.model_report = {}
        if target_col and target_col in self.df.columns:
            self.model_report = train_and_evaluate(self.df, target_col)
        self.narrative = narrate_insights(
            self.eda, self.model_report, self.title, self.llm_model
        )
        return {
            'df':           self.df,
            'eda':          self.eda,
            'model_report': self.model_report,
            'narrative':    self.narrative,
        }

    def save_chart(self, out_path: str = 'insight_report.png') -> str:
        """
        Save a 2\u00d72 dashboard:
          - Top-left:  histogram of first numeric column
          - Top-right: correlation heatmap (numeric columns)
          - Bot-left:  horizontal bar chart of first categorical column
          - Bot-right: scatter of first two numeric columns
        """
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt

        if self.df is None:
            raise RuntimeError('Call run() before save_chart().')

        df       = self.df
        num_cols = self.eda.get('numeric_cols', [])
        cat_cols = self.eda.get('cat_cols', [])

        fig, axes = plt.subplots(2, 2, figsize=(13, 9))
        fig.suptitle(f'{self.title} \u2014 Insight Report', fontsize=14)

        # Top-left: distribution of first numeric column
        ax = axes[0, 0]
        if num_cols:
            col = num_cols[0]
            ax.hist(df[col].dropna(), bins=20, edgecolor='white', color='steelblue')
            ax.set_title(f'Distribution: {col}')
            ax.set_xlabel(col); ax.set_ylabel('Count')

        # Top-right: correlation heatmap
        ax = axes[0, 1]
        if len(num_cols) >= 2:
            corr = df[num_cols].corr()
            ax.imshow(corr, cmap='RdBu', vmin=-1, vmax=1)
            ticks = range(len(num_cols))
            ax.set_xticks(ticks); ax.set_xticklabels(num_cols, rotation=45, ha='right')
            ax.set_yticks(ticks); ax.set_yticklabels(num_cols)
            ax.set_title('Correlation Heatmap')
            for i in range(len(num_cols)):
                for j in range(len(num_cols)):
                    ax.text(j, i, f'{corr.iloc[i, j]:.2f}',
                            ha='center', va='center', fontsize=7)

        # Bottom-left: top category counts
        ax = axes[1, 0]
        if cat_cols:
            col = cat_cols[0]
            vc  = df[col].value_counts().head(8)
            ax.barh(vc.index.tolist()[::-1], vc.values[::-1], color='coral')
            ax.set_title(f'Counts by {col}')
            ax.set_xlabel('Count')

        # Bottom-right: scatter of first two numeric columns
        ax = axes[1, 1]
        if len(num_cols) >= 2:
            ax.scatter(df[num_cols[0]].dropna(), df[num_cols[1]].dropna(),
                       alpha=0.35, color='teal', s=18)
            ax.set_title(f'{num_cols[0]} vs {num_cols[1]}')
            ax.set_xlabel(num_cols[0]); ax.set_ylabel(num_cols[1])

        plt.tight_layout()
        fig.savefig(out_path, bbox_inches='tight', dpi=100)
        plt.close('all')
        return out_path

## Your Pipeline

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# TODO: engine = InsightEngine(title='Retail Sales 2023')
# TODO: result = engine.run(make_sample_data(300), target_col='revenue')
# TODO: engine.save_chart('insight_report.png')

# TODO: print('=== AI Executive Summary ===')
# TODO: print(engine.narrative)

# TODO: print('\n=== Model Report ===')
# TODO: mr = engine.model_report
# TODO: print(f"Target: {mr['target']}")
# TODO: print(f"CV R\u00b2: {mr['cv_r2']['mean']:.4f} \u00b1 {mr['cv_r2']['std']:.4f}")
# TODO: print(f"Test R\u00b2: {mr['test_r2']:.4f}   RMSE: {mr['test_rmse']:.2f}")
# TODO: top = sorted(mr['coefficients'].items(), key=lambda kv: abs(kv[1]), reverse=True)[0]
# TODO: print(f"Strongest predictor: {top[0]} (coef={top[1]:.4f})")

## Checks

In [ ]:
import os


def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: engine exists and has been run
    try:
        assert 'engine' in globals(), 'engine not defined'
        assert isinstance(engine, InsightEngine), \
            'engine must be an InsightEngine instance'
        assert engine.df is not None, 'engine.df is None — call engine.run() first'
        passed += 1; print(f'\u2705 Check 1: InsightEngine created and run()')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: cleaned DataFrame has 300 rows (approx) and no numeric nulls
    try:
        assert len(engine.df) >= 290, \
            f'expected ~300 rows, got {len(engine.df)}'
        num_nulls = engine.df.select_dtypes(include='number').isnull().sum().sum()
        assert num_nulls == 0, f'{num_nulls} numeric nulls remain after cleaning'
        passed += 1; print(f'\u2705 Check 2: df has {len(engine.df)} rows, 0 numeric nulls')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: model trained on revenue — test_r2 > 0.5
    try:
        mr = engine.model_report
        assert 'test_r2' in mr, "model_report missing 'test_r2'"
        assert mr['test_r2'] > 0.5, \
            f"test_r2={mr['test_r2']} should be > 0.5"
        passed += 1; print(f"\u2705 Check 3: test_r2={mr['test_r2']} > 0.5")
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: narrative is a non-empty string
    try:
        assert isinstance(engine.narrative, str) and len(engine.narrative) > 20
        passed += 1; print(f'\u2705 Check 4: narrative ({len(engine.narrative)} chars)')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: insight_report.png saved
    try:
        assert os.path.exists('insight_report.png'), \
            'insight_report.png not found — call engine.save_chart()'
        assert os.path.getsize('insight_report.png') > 1000
        passed += 1; print(f'\u2705 Check 5: insight_report.png saved')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete! Section 3 done.')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()